# 第 16 天：Alpha101导论

> 所属阶段：从“经典单因子”进入“公式化 Alpha 工厂”
> 今日主题：Alpha101导论
> 必做：阅读论文
> 选做：整理公式
> 目标产出：Alpha101笔记

今天开始，你会从“我知道 PE、ROE、动量这些传统因子”切换到另一个世界：公式化 Alpha。

传统因子像是几种常见食材：价值、质量、动量、波动率、流动性。  
Alpha101 更像是一套厨房语言：`rank`、`delay`、`delta`、`correlation`、`ts_rank`、`adv20`、`vwap` 这些算子可以拼出很多味道不同的信号。

这一阶段最重要的不是背公式，而是学会三件事：

1. 看懂公式里的每个算子在说什么。
2. 把公式翻译成稳定、可检查、无未来函数的 Python。
3. 用因子研究流程判断它有没有研究价值，而不是只看公式是否“高级”。



## 0. 今天你要真正学会什么？

第 16 天不是急着复制公式。今天的目标是搭好“读公式、翻译公式、检验公式”的工作台。

你需要把 Alpha101 看成三层东西：

1. **语言层**：公式由横截面算子和时间序列算子组成。
2. **数据层**：`open`、`high`、`low`、`close`、`volume`、`vwap`、`returns` 必须在同一日期、同一股票池上对齐。
3. **研究层**：公式算出来只是候选因子，后面还要经过 IC、分层、换手、相关性、稳定性检验。

今天结束时，你应该能做到：

- 看到一个 Alpha101 公式，先拆算子，而不是被括号吓住。
- 判断公式里有没有明显未来函数风险。
- 写出一套最小可用的 Alpha101 算子库。
- 把任意一个公式转成因子矩阵，并用 Rank IC 做第一轮检查。

## 1. 先建立直觉：Alpha101 像一门“因子乐高语言”

传统因子常常是一个财务或市场指标：PB、ROE、20 日动量、换手率。

Alpha101 的风格更像：

- 用 `rank` 把横截面绝对值变成相对强弱。
- 用 `delta` 捕捉变化，而不是只看水平。
- 用 `delay` 明确时间顺序。
- 用 `correlation` 观察两个变量在短窗口内是否一起变。
- 用 `ts_rank` 判断今天在过去一段时间里算不算极端。

所以 Alpha101 不是“101 个神秘公式”，而是一套高频组合语言。  
你真正要学的是这门语言背后的表达方式：价格行为、成交量行为、相对强弱、短期反转、趋势延续、量价背离。

## 2. 今天的核心问题

### 问题 1：公式里的 `rank` 到底是在排什么？

大多数时候，`rank(x)` 指的是横截面排名：同一天把所有股票的 `x` 值放在一起排序。

这意味着：

- 它不关心某只股票自己的绝对水平。
- 它关心这只股票今天在全市场里相对靠前还是靠后。
- 它天然适合选股，因为选股本来就是横截面比较。

### 问题 2：`ts_rank` 和 `rank` 有什么区别？

`rank` 是“今天所有股票之间比”。  
`ts_rank` 是“某只股票今天和自己的过去比”。

举例：

- `rank(volume)`：今天哪只股票成交量相对最大。
- `ts_rank(volume, 20)`：这只股票今天成交量在过去 20 天里是不是很高。

### 问题 3：为什么 Alpha101 经常把价格和成交量混在一起？

价格反映结果，成交量反映参与程度。

一个价格上涨但成交量萎缩的股票，和一个价格上涨且成交量放大的股票，市场含义可能完全不同。  
Alpha101 里大量公式都在寻找这种“价格行为”和“交易行为”之间的错位。

## 3. Alpha101 研究流程全景


读取公式
  -> 拆出字段：open/high/low/close/volume/vwap/returns
  -> 拆出算子：rank/delay/delta/correlation/ts_rank
  -> 统一数据频率和股票池
  -> 转成因子矩阵
  -> 清洗缺失值和极端值
  -> IC 检验
  -> 分层回测
  -> 行业、市值中性化
  -> 相关性和冗余检查
  -> 加入因子库或丢弃


这条流程很朴素，但它是公式化 Alpha 的安全带。

## 4. 准备一套可控的模拟行情数据

真实数据当然更好，但初学阶段先用模拟数据有两个好处：

1. 你能专注于公式翻译，不会被停牌、复权、缺失、股票池变化打断。
2. 你可以反复运行，确认代码逻辑稳定。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(202620)

dates = pd.bdate_range("2022-01-03", periods=360)
assets = [f"S{i:03d}" for i in range(1, 51)]

industries = pd.Series(
    np.random.choice(["消费", "科技", "制造", "医药", "金融"], size=len(assets)),
    index=assets,
    name="industry",
)
size_score = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="size_score")
asset_beta = pd.Series(np.random.uniform(0.75, 1.35, len(assets)), index=assets, name="beta")
asset_drift = pd.Series(np.random.normal(0.00015, 0.00018, len(assets)), index=assets, name="drift")

market_return = np.random.normal(0.00025, 0.010, len(dates))
style_shock = np.random.normal(0, 0.004, (len(dates), len(assets)))
idiosyncratic = np.random.normal(0, 0.018, (len(dates), len(assets)))

daily_return = (
    market_return[:, None] * asset_beta.values[None, :]
    + asset_drift.values[None, :]
    + style_shock
    + idiosyncratic
)

close = pd.DataFrame(
    30 * np.exp(np.cumsum(daily_return, axis=0)),
    index=dates,
    columns=assets,
)

overnight = pd.DataFrame(
    np.random.normal(0, 0.006, close.shape),
    index=dates,
    columns=assets,
)
open_ = close.shift(1) * (1 + overnight)
open_.iloc[0] = close.iloc[0] * (1 + overnight.iloc[0])

intraday_span = pd.DataFrame(
    np.random.uniform(0.002, 0.035, close.shape),
    index=dates,
    columns=assets,
)
high = pd.DataFrame(
    np.maximum(open_.to_numpy(), close.to_numpy()) * (1 + intraday_span.to_numpy()),
    index=dates,
    columns=assets,
)
low = pd.DataFrame(
    np.minimum(open_.to_numpy(), close.to_numpy()) * (1 - intraday_span.to_numpy()),
    index=dates,
    columns=assets,
)

base_volume = (900_000 * np.exp(size_score.values))[None, :]
volume = pd.DataFrame(
    base_volume
    * np.random.lognormal(mean=0, sigma=0.45, size=close.shape)
    * (1 + close.pct_change().fillna(0).abs().to_numpy() * 12),
    index=dates,
    columns=assets,
)
vwap = (open_ + high + low + close) / 4
returns = close.pct_change()
adv20 = volume.rolling(20).mean()
future_5d = close.shift(-5) / close - 1

print("数据形状：")
print({
    "close": close.shape,
    "open": open_.shape,
    "high": high.shape,
    "low": low.shape,
    "volume": volume.shape,
    "future_5d": future_5d.shape,
})
print("\n行业分布：")
print(industries.value_counts())


## 5. 搭建 Alpha101 最小算子库

下面这套算子不是最终工业版，但足够你完成第 16-20 天的学习。

重点不是函数数量，而是接口统一：

- 输入都是日期 x 股票的 DataFrame。
- 输出也尽量保持同样形状。
- 所有时间序列窗口都只看过去和今天。
- 所有横截面操作都在同一天股票之间完成。


In [ ]:
def cs_rank(df: pd.DataFrame) -> pd.DataFrame:
    """横截面排名：每天在所有股票之间排序，输出 0 到 1 附近的百分位。"""
    return df.rank(axis=1, pct=True)


def delay(df: pd.DataFrame, n: int = 1) -> pd.DataFrame:
    """向后取 n 期，避免把今天之后的信息放进今天。"""
    return df.shift(n)


def delta(df: pd.DataFrame, n: int = 1) -> pd.DataFrame:
    """当前值减去 n 期前的值。"""
    return df - delay(df, n)


def ts_mean(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).mean()


def ts_sum(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).sum()


def ts_std(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).std()


def ts_min(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).min()


def ts_max(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).max()


def ts_rank(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """时间序列排名：今天的值在过去 window 天里排第几。"""
    return df.rolling(window).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1],
        raw=False,
    )


def ts_argmax(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """过去 window 天最大值出现的位置，1 表示窗口第一天，window 表示今天。"""
    return df.rolling(window).apply(lambda x: np.argmax(x) + 1, raw=True)


def corr(a: pd.DataFrame, b: pd.DataFrame, window: int) -> pd.DataFrame:
    return a.rolling(window).corr(b)


def cov(a: pd.DataFrame, b: pd.DataFrame, window: int) -> pd.DataFrame:
    return a.rolling(window).cov(b)


def signed_power(df: pd.DataFrame, power: float) -> pd.DataFrame:
    return np.sign(df) * (df.abs() ** power)


def decay_linear(df: pd.DataFrame, window: int) -> pd.DataFrame:
    weights = np.arange(1, window + 1, dtype=float)
    weights = weights / weights.sum()
    return df.rolling(window).apply(lambda x: np.dot(x, weights), raw=True)


def safe_clean(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan)


def neutralize_by_group(factor: pd.DataFrame, groups: pd.Series) -> pd.DataFrame:
    """简化版行业中性化：每个交易日、每个行业内减去行业均值。"""
    result = pd.DataFrame(index=factor.index, columns=factor.columns, dtype=float)
    for _, cols in groups.groupby(groups).groups.items():
        cols = list(cols)
        block = factor[cols]
        result[cols] = block.sub(block.mean(axis=1), axis=0)
    return result


def calc_rank_ic(factor: pd.DataFrame, label: pd.DataFrame) -> pd.Series:
    """按日期计算 Rank IC。"""
    factor = safe_clean(factor)
    label = label.reindex_like(factor)
    return factor.rank(axis=1).corrwith(label.rank(axis=1), axis=1)


def factor_report(factor: pd.DataFrame, label: pd.DataFrame, name: str) -> pd.Series:
    """给单个因子生成一个轻量研究摘要。"""
    f = safe_clean(factor)
    ic = calc_rank_ic(f, label).dropna()
    q = f.rank(axis=1, pct=True)
    long_leg = label.where(q >= 0.8).mean(axis=1)
    short_leg = label.where(q <= 0.2).mean(axis=1)
    ls = (long_leg - short_leg).dropna()
    turnover = q.ge(0.8).astype(float).diff().abs().mean(axis=1).dropna()

    return pd.Series({
        "factor": name,
        "ic_mean": ic.mean(),
        "ic_ir": ic.mean() / ic.std() if ic.std() != 0 else np.nan,
        "ic_positive_ratio": (ic > 0).mean(),
        "long_short_mean": ls.mean(),
        "long_short_win_rate": (ls > 0).mean(),
        "top_bucket_turnover": turnover.mean(),
        "valid_days": len(ic),
    })


def describe_factor_set(factors: dict[str, pd.DataFrame], label: pd.DataFrame) -> pd.DataFrame:
    rows = [factor_report(factor, label, name) for name, factor in factors.items()]
    return pd.DataFrame(rows).set_index("factor").sort_values("ic_mean", ascending=False)


print("核心算子已准备好：cs_rank、delay、delta、ts_rank、corr、cov、neutralize、IC report")


## 6. 把一个公式翻译成 Python 的三步法

假设你看到一个教学公式：


-1 * correlation(rank(open), rank(volume), 10)


不要一上来就写一长行。先拆成三层：

1. `rank(open)`：每天把股票按开盘价相对水平排序。
2. `rank(volume)`：每天把股票按成交量相对水平排序。
3. `correlation(..., 10)`：看过去 10 天两者是否同步。

然后再乘以 `-1`，表示如果价格相对高和成交量相对高高度同步，公式倾向给低分。


In [ ]:
rank_open = cs_rank(open_)
rank_volume = cs_rank(volume)
toy_alpha = -1 * corr(rank_open, rank_volume, 10)

toy_ic = calc_rank_ic(toy_alpha, future_5d)
print(toy_alpha.tail(3).round(3))
print("\nToy Alpha IC：")
print(toy_ic.describe().round(4))


## 7. 公式翻译检查表

你以后每复现一个 Alpha，都建议按这个顺序检查：

| 检查项 | 你要问自己的问题 |
|---|---|
| 字段 | `open/high/low/close/volume/vwap/returns` 是否都按日期和股票对齐？ |
| 窗口 | 所有 rolling 窗口是否只使用过去信息？ |
| 排名方向 | 高值到底代表买入信号，还是卖出信号？ |
| 缺失值 | 前 N 天自然缺失是否保留？ |
| 极端值 | 是否存在除以 0、无穷大、异常成交量？ |
| 标签 | 因子日期和未来收益标签是否对齐？ |
| 评价 | 是否至少看 IC、ICIR、胜率和换手？ |

## 8. 建一个“算子词典”

很多同学学 Alpha101 会卡在缩写上。你可以把算子当成一张词典。


In [ ]:
operator_dictionary = pd.DataFrame([
    ["rank(x)", "横截面", "同一天股票之间排序", "选股相对强弱"],
    ["ts_rank(x, n)", "时间序列", "今天在过去 n 天里的位置", "判断是否极端"],
    ["delay(x, n)", "时间序列", "n 天前的值", "避免未来函数"],
    ["delta(x, n)", "时间序列", "今天减 n 天前", "变化、动量、反转"],
    ["correlation(x, y, n)", "时间序列", "过去 n 天相关性", "量价关系"],
    ["covariance(x, y, n)", "时间序列", "过去 n 天协方差", "共同波动"],
    ["adv20", "衍生字段", "20 日平均成交量", "判断成交量是否异常"],
    ["vwap", "价格字段", "成交量加权平均价", "接近交易成本和成交位置"],
], columns=["算子", "类型", "含义", "研究用途"])

print(operator_dictionary)


## 9. 第一个完整小实验：公式、IC、分层、换手


In [ ]:
alpha_price_volume = -1 * corr(cs_rank(open_), cs_rank(volume), 10)

report = factor_report(alpha_price_volume, future_5d, "price_volume_corr")
print(report.drop(labels=["factor"]).astype(float).round(4))

q = alpha_price_volume.rank(axis=1, pct=True)
group_return = pd.DataFrame({
    "Q1_low": future_5d.where(q <= 0.2).mean(axis=1),
    "Q5_high": future_5d.where(q >= 0.8).mean(axis=1),
})
group_return["long_short"] = group_return["Q5_high"] - group_return["Q1_low"]

print("\n分组收益摘要：")
print(group_return.describe().round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

calc_rank_ic(alpha_price_volume, future_5d).dropna().cumsum().plot(
    ax=axes[0],
    title="累计 Rank IC",
)
group_return["long_short"].dropna().cumsum().plot(
    ax=axes[1],
    title="多空组合累计收益",
)

plt.tight_layout()
plt.show()
plt.close()


## 10. 参数敏感性：窗口不是装饰品

Alpha101 公式里有很多窗口：3、5、6、10、20、60、250。  
窗口的变化会改变因子的性格。


In [ ]:
rows = []
for window in [3, 5, 10, 20, 40]:
    factor = -1 * corr(cs_rank(open_), cs_rank(volume), window)
    rows.append(factor_report(factor, future_5d, f"window_{window}"))

window_report = pd.DataFrame(rows).set_index("factor")
print(window_report[["ic_mean", "ic_ir", "ic_positive_ratio", "top_bucket_turnover"]].round(4))


## 11. 行业中性化：公式有效还是行业暴露有效？

公式化 Alpha 很容易混入行业结构。

如果某个公式看起来有效，但本质只是“最近科技股涨得多”，那它未必是可持续 Alpha。


In [ ]:
raw_factor = -1 * corr(cs_rank(open_), cs_rank(volume), 10)
industry_neutral_factor = neutralize_by_group(raw_factor, industries)

compare = pd.DataFrame({
    "raw": factor_report(raw_factor, future_5d, "raw"),
    "industry_neutral": factor_report(industry_neutral_factor, future_5d, "industry_neutral"),
}).T

print(compare[["ic_mean", "ic_ir", "long_short_mean", "top_bucket_turnover"]].round(4))


## 12. Alpha101 笔记应该怎么写？

你的 Alpha101 笔记不应该只抄公式。建议每个公式都按下面模板记录：

### 12.1 公式原貌

写下公式，但不要急着解释。

### 12.2 字段解释

列出用到的字段：

- 价格：`open`、`high`、`low`、`close`、`vwap`
- 成交：`volume`、`adv20`
- 收益：`returns`

### 12.3 算子解释

列出所有算子，并写出它们发生在横截面还是时间序列。

### 12.4 因子直觉

用一句自然语言说明它可能在捕捉什么。

例如：  
“这个因子试图寻找开盘价相对位置和成交量相对位置在短期内高度同步的股票，并对这种同步关系取反。”

### 12.5 实证结果

至少记录：

- Rank IC 均值
- ICIR
- IC 为正比例
- 多空平均收益
- 换手率
- 是否行业中性化后仍有效

## 13. 今天最容易踩的 10 个坑

### 坑 1：把 `rank` 当成时间序列排名

Alpha101 里很多 `rank` 是横截面排名。  
如果你写成每只股票自己的历史排名，因子含义会完全变掉。

### 坑 2：把 `delay` 写反

`delay(close, 1)` 是昨天的 close。  
如果你误写成未来一天，回测会非常漂亮，但那是假的。

### 坑 3：不保留自然缺失

20 日窗口前 19 天没有结果是正常的。  
不要为了“数据整齐”把它填成 0。

### 坑 4：先看收益再改公式

如果你看到结果不好就不断改公式，很容易变成过拟合。  
要先写清楚改造理由，再做实验。

### 坑 5：只看单个 Alpha

Alpha101 的价值不只在单公式，而在大量候选因子的组合、筛选、去冗余。

### 坑 6：忘记交易成本

很多短周期公式换手很高。  
纸面 IC 不错，扣掉成本后可能没有意义。

### 坑 7：忽略极端成交量

成交量异常会让相关性、排名、协方差都变得很敏感。

### 坑 8：没有统一数据频率

如果某些字段是复权价，某些字段是原始价，公式会混乱。

### 坑 9：把公式当神谕

公式只是候选假设，不是结论。

### 坑 10：没有写研究日志

公式化研究最容易丢失上下文。  
你必须记录每次改造的理由、参数和结果。

## 14. 今日动手作业

### 作业 A：整理 Alpha101 词典

把今天出现的算子写成一张表：

- 算子名
- 输入
- 输出
- 横截面还是时间序列
- 容易写错的地方

### 作业 B：改造 Toy Alpha

把窗口从 10 改成 5、20、40，记录 IC 和换手变化。

### 作业 C：做一次行业中性化

比较中性化前后的 IC。  
如果变化很大，写一句解释。

### 作业 D：写 Alpha101 笔记模板

创建一个固定模板，以后第 17-20 天每个 Alpha 都按同样格式记录。

## 15. 面试式自测

### 问 1：`rank` 和 `ts_rank` 的本质区别是什么？

答案：`rank` 是同一天股票之间比较，`ts_rank` 是同一只股票在过去时间窗口中比较。

### 问 2：为什么公式化 Alpha 必须检查换手？

答案：因为很多短周期信号频繁变化，未扣成本时看起来有效，实际交易后可能消失。

### 问 3：为什么要做行业中性化？

答案：为了判断因子收益来自公式本身，还是来自行业暴露。

### 问 4：模拟数据结果能证明公式有效吗？

答案：不能。模拟数据只用于验证代码逻辑，真实有效性必须用真实历史数据检验。

## 16. 今日复盘模板


今天我学会的 3 个算子：
1.
2.
3.

我最容易写错的地方：

我今天复现的第一个公式：

它的经济直觉：

IC 检验结果：

行业中性化后变化：

下一步我要改进：


## 17. 明天预告：Alpha101 复现 1-5 号

明天会真正进入公式复现。  
第 1-5 号 Alpha 会让你看到三类典型思想：

- 波动替代价格后的短期极值。
- 开盘价、成交量和日内收益的关系。
- 低价、VWAP、收盘价之间的相对位置。

## 18. 一句话收尾

Alpha101 的门槛不是公式复杂，而是你能不能把公式拆成可检查的研究流程。

## 19. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本和严格的样本外检验。
